# vWAN Routing Intent + NAT Gateway Bypass Test

**Frage:** Kann ein UDR mit Service Tag `WindowsVirtualDesktop` + Next-Hop `Internet` (NAT Gateway) den Azure Firewall umgehen, wenn vWAN Routing Intent aktiv ist?

**Setup:**
- vWAN Secured Hub mit Azure Firewall + Routing Intent (Private + Internet → FW)
- AVD Spoke VNet mit UDR: `WindowsVirtualDesktop` → Internet
- NAT Gateway auf dem AVD-Subnet

**Erwartetes Ergebnis:**
- Default Traffic (0/0) → Firewall PIP
- WVD Service Tag Traffic → NAT Gateway PIP (falls Bypass funktioniert)

## Variablen

In [2]:
export PREFIX=cptdazavdvwan
export RG=rg-${PREFIX}
export SUB=$(az account show --query id -o tsv)
export VM=vm-avd-${PREFIX}
export NIC=nic-avd-${PREFIX}
echo "RG=$RG SUB=$SUB VM=$VM NIC=$NIC"

RG=rg-cptdazavdvwan SUB=ff0bb075-6c44-44ee-bb64-d46ce828c62f VM=vm-avd-cptdazavdvwan NIC=nic-avd-cptdazavdvwan


## 1. Public IPs abrufen

**Erwartung:** Zwei unterschiedliche PIPs — eine für den Firewall, eine für das NAT Gateway.

In [ ]:
# Outputs der beiden getrennten Deployments (platform + avd) zusammenführen
{ az deployment group show -g $RG -n platform --query properties.outputs -o json
  az deployment group show -g $RG -n avd --query properties.outputs -o json
} | jq -s 'add' | tee output/deployment-outputs.json

{
  "avdVmName": {
    "type": "String",
    "value": "vm-avd-cptdazavdvwan"
  },
  "bastionName": {
    "type": "String",
    "value": "bas-cptdazavdvwan"
  },
  "firewallPrivateIp": {
    "type": "String",
    "value": "10.0.64.4"
  },
  "firewallPublicIp": {
    "type": "String",
    "value": "4.223.154.110"
  },
  "hostPoolName": {
    "type": "String",
    "value": "hp-cptdazavdvwan"
  },
  "natGatewayPublicIp": {
    "type": "String",
    "value": "20.91.239.174"
  },
  "routeTableId": {
    "type": "String",
    "value": "/subscriptions/ff0bb075-6c44-44ee-bb64-d46ce828c62f/resourceGroups/rg-cptdazavdvwan/providers/Microsoft.Network/routeTables/rt-avd-cptdazavdvwan"
  },
  "routeTableName": {
    "type": "String",
    "value": "rt-avd-cptdazavdvwan"
  }
}


## 2. Route Table anzeigen

**Erwartung:**
- Route `avd-wvd-direct-internet`: Prefix=`WindowsVirtualDesktop`, NextHop=`Internet`
- Route `test-udr-more-specific`: Prefix=`10.99.0.0/16`, NextHop=`None`

In [5]:
az network route-table show -g $RG -n rt-avd-${PREFIX} --query routes -o json | tee output/route-table.json

[
  {
    "addressPrefix": "WindowsVirtualDesktop",
    "etag": "W/\"5cf4d090-e8be-4048-b5c8-7a68c2085c14\"",
    "hasBgpOverride": false,
    "id": "/subscriptions/ff0bb075-6c44-44ee-bb64-d46ce828c62f/resourceGroups/rg-cptdazavdvwan/providers/Microsoft.Network/routeTables/rt-avd-cptdazavdvwan/routes/avd-wvd-direct-internet",
    "name": "avd-wvd-direct-internet",
    "nextHopType": "Internet",
    "provisioningState": "Succeeded",
    "resourceGroup": "rg-cptdazavdvwan",
    "type": "Microsoft.Network/routeTables/routes"
  },
  {
    "addressPrefix": "10.99.0.0/16",
    "etag": "W/\"5cf4d090-e8be-4048-b5c8-7a68c2085c14\"",
    "hasBgpOverride": false,
    "id": "/subscriptions/ff0bb075-6c44-44ee-bb64-d46ce828c62f/resourceGroups/rg-cptdazavdvwan/providers/Microsoft.Network/routeTables/rt-avd-cptdazavdvwan/routes/test-udr-more-specific",
    "name": "test-udr-more-specific",
    "nextHopType": "None",
    "provisioningState": "Succeeded",
    "resourceGroup": "rg-cptdazavdvwan",
    "ty

## 3. Effective Routes (DER SCHLÜSSELTEST)

**Wenn Bypass funktioniert:**
- `0.0.0.0/0` → Source=VirtualNetworkGateway (Routing Intent)
- WindowsVirtualDesktop IP-Ranges → Source=User, NextHop=Internet
- `10.99.0.0/16` → Source=User, NextHop=None

**Wenn Bypass NICHT funktioniert:**
- `0.0.0.0/0` → VirtualNetworkGateway (Routing Intent überschreibt alle Internet-UDRs)
- Keine User-Routen mit NextHop=Internet sichtbar

In [6]:
az network nic show-effective-route-table -g $RG -n $NIC -o json | tee output/effective-routes.json

{/ Finished ..
  "value": [
    {
      "addressPrefix": [
        "10.1.0.0/16"
      ],
      "disableBgpRoutePropagation": false,
      "nextHopIpAddress": [],
      "nextHopType": "VnetLocal",
      "source": "Default",
      "state": "Active"
    },
    {
      "addressPrefix": [
        "10.0.0.0/16"
      ],
      "disableBgpRoutePropagation": false,
      "nextHopIpAddress": [],
      "nextHopType": "VNetPeering",
      "source": "Default",
      "state": "Active"
    },
    {
      "addressPrefix": [
        "192.168.0.0/16"
      ],
      "disableBgpRoutePropagation": false,
      "nextHopIpAddress": [
        "10.0.64.4"
      ],
      "nextHopType": "VirtualNetworkGateway",
      "source": "VirtualNetworkGateway",
      "state": "Active"
    },
    {
      "addressPrefix": [
        "0.0.0.0/0"
      ],
      "disableBgpRoutePropagation": false,
      "nextHopIpAddress": [
        "10.0.64.4"
      ],
      "nextHopType": "VirtualNetworkGateway",
      "source": "VirtualNet

In [7]:
# Gefiltert: Nur User-definierte Routen
az network nic show-effective-route-table -g $RG -n $NIC --query "value[?source=='User']" -o table

DisableBgpRoutePropagation    Name                     NextHopType    Source    State
----------------------------  -----------------------  -------------  --------  -------
False                         avd-wvd-direct-internet  Internet       User      Active
False                         test-udr-more-specific   None           User      Active


In [8]:
# Gefiltert: Default Route (0/0) - von Routing Intent injiziert
az network nic show-effective-route-table -g $RG -n $NIC --query "value[?contains(addressPrefix[0],'0.0.0.0/0')]" -o json

[| Finished ..
  {
    "addressPrefix": [
      "0.0.0.0/0"
    ],
    "disableBgpRoutePropagation": false,
    "nextHopIpAddress": [
      "10.0.64.4"
    ],
    "nextHopType": "VirtualNetworkGateway",
    "source": "VirtualNetworkGateway",
    "state": "Active"
  }
]


## 4. Outbound IP Test (Default Route)

**Erwartung:** Die VM zeigt die Firewall-PIP als Outbound-IP, weil Routing Intent 0/0 → FW leitet.

Falls Timeout: FW blockiert HTTP zu `ifconfig.me` (keine passende App Rule).

In [9]:
az vm run-command invoke -g $RG -n $VM --command-id RunPowerShellScript \
  --scripts "(Invoke-WebRequest -Uri http://ifconfig.me/ip -UseBasicParsing -TimeoutSec 15).Content" \
  -o json | tee output/outbound-ip.json

{| Finished ..
  "value": [
    {
      "code": "ComponentStatus/StdOut/succeeded",
      "displayStatus": "Provisioning succeeded",
      "level": "Info",
      "message": ""
    },
    {
      "code": "ComponentStatus/StdErr/succeeded",
      "displayStatus": "Provisioning succeeded",
      "level": "Info",
      "message": "Invoke-WebRequest : Action: Deny. Reason: No rule matched. Proceeding with default action.\nAt C:\\Packages\\Plugins\\Microsoft.CPlat.Core.RunCommandWindows\\1.1.22\\Downloads\\script2.ps1:1 char:2\n+ (Invoke-WebRequest -Uri http://ifconfig.me/ip -UseBasicParsing -Timeo ...\n+  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~\n    + CategoryInfo          : InvalidOperation: (System.Net.HttpWebRequest:HttpWebRequest) [Invoke-WebRequest], WebExc \n   eption\n    + FullyQualifiedErrorId : WebCmdletWebResponseException,Microsoft.PowerShell.Commands.InvokeWebRequestCommand"
    }
  ]
}


## 5. WVD Endpoint Konnektivität

**Erwartung:** TCP 443 zu WVD-Endpunkten funktioniert — entweder direkt via NAT GW (Bypass) oder via FW (Application Rule).

In [10]:
az vm run-command invoke -g $RG -n $VM --command-id RunPowerShellScript \
  --scripts "Test-NetConnection rdweb.wvd.microsoft.com -Port 443 | Format-List" \
  -o json | tee output/wvd-connectivity.json

{| Finished ..
  "value": [
    {
      "code": "ComponentStatus/StdOut/succeeded",
      "displayStatus": "Provisioning succeeded",
      "level": "Info",
      "message": "ComputerName     : rdweb.wvd.microsoft.com\nRemoteAddress    : 40.64.146.225\nRemotePort       : 443\nInterfaceAlias   : Ethernet\nSourceAddress    : 10.1.1.4\nTcpTestSucceeded : True\n\n\n"
    },
    {
      "code": "ComponentStatus/StdErr/succeeded",
      "displayStatus": "Provisioning succeeded",
      "level": "Info",
      "message": ""
    }
  ]
}


In [11]:
az vm run-command invoke -g $RG -n $VM --command-id RunPowerShellScript \
  --scripts "Test-NetConnection rdbroker.wvd.microsoft.com -Port 443 | Format-List" \
  -o json | tee output/wvd-broker-connectivity.json

{/ Finished ..
  "value": [
    {
      "code": "ComponentStatus/StdOut/succeeded",
      "displayStatus": "Provisioning succeeded",
      "level": "Info",
      "message": "ComputerName     : rdbroker.wvd.microsoft.com\nRemoteAddress    : 40.64.144.35\nRemotePort       : 443\nInterfaceAlias   : Ethernet\nSourceAddress    : 10.1.1.4\nTcpTestSucceeded : True\n\n\n"
    },
    {
      "code": "ComponentStatus/StdErr/succeeded",
      "displayStatus": "Provisioning succeeded",
      "level": "Info",
      "message": ""
    }
  ]
}


## 6. NAT Gateway Metrics

**Erwartung (Bypass funktioniert):** ByteCount > 0, SNATConnectionCount > 0

**Erwartung (Bypass funktioniert NICHT):** Alle Werte = 0 oder null

In [12]:
NATGW_ID=$(az network nat gateway show -g $RG -n natgw-avd-${PREFIX} --query id -o tsv)
az monitor metrics list --resource "$NATGW_ID" --metric ByteCount --interval PT5M -o json | tee output/natgw-metrics.json

{
  "cost": 59,
  "interval": "PT5M",
  "namespace": "Microsoft.Network/natGateways",
  "resourceregion": "swedencentral",
  "timespan": "2026-06-10T05:39:53Z/2026-06-10T06:39:53Z",
  "value": [
    {
      "displayDescription": "Total number of Bytes transmitted within time period",
      "errorCode": "Success",
      "id": "/subscriptions/ff0bb075-6c44-44ee-bb64-d46ce828c62f/resourceGroups/rg-cptdazavdvwan/providers/Microsoft.Network/natGateways/natgw-avd-cptdazavdvwan/providers/Microsoft.Insights/metrics/ByteCount",
      "name": {
        "localizedValue": "Bytes",
        "value": "ByteCount"
      },
      "resourceGroup": "rg-cptdazavdvwan",
      "timeseries": [],
      "type": "Microsoft.Insights/metrics",
      "unit": "Bytes"
    }
  ]
}


### 6.1 Beweis: RDP-Traffic via NAT Gateway (nicht Firewall)

**Methode:**
1. WVD-Endpoint IP auflösen → prüfen ob in Effective Routes als User-Route (NextHop=Internet)
2. NAT Gateway SNAT Connection Count > 0 → Traffic fliesst durch NAT GW
3. Azure FW Log Query → WVD-Ziel-IPs sollten NICHT im FW-Log erscheinen (= Bypass bestätigt)

In [13]:
# 1) WVD Gateway IP auflösen und in Effective Routes prüfen
echo "=== WVD Gateway IP-Adressen ==="
WVD_IP=$(az vm run-command invoke -g $RG -n $VM --command-id RunPowerShellScript \
  --scripts "(Resolve-DnsName rdbroker.wvd.microsoft.com -Type A).IPAddress | Select-Object -First 3" \
  -o json | jq -r '.value[0].message' | grep -oP '\d+\.\d+\.\d+\.\d+' | head -3)
echo "$WVD_IP"

echo ""
echo "=== Effective Route für diese IPs (sollte Source=User, NextHop=Internet sein) ==="
FIRST_IP=$(echo "$WVD_IP" | head -1)
az network nic show-effective-route-table -g $RG -n $NIC -o json | \
  jq --arg ip "$FIRST_IP" '[.value[] | select(.source=="User" and .nextHopType=="Internet") | {prefix: .addressPrefix[0], nextHopType, source}] | .[0:3]'

echo ""
echo "=== NAT Gateway SNAT Connections (letzte 30 Min) ==="
NATGW_ID=$(az network nat gateway show -g $RG -n natgw-avd-${PREFIX} --query id -o tsv)
az monitor metrics list --resource "$NATGW_ID" \
  --metric SNATConnectionCount \
  --interval PT5M \
  --start-time $(date -u -d '30 minutes ago' +%Y-%m-%dT%H:%M:%SZ) \
  -o json | jq '[.value[0].timeseries[0].data[] | select(.total > 0) | {time: .timeStamp, connections: .total}]'

echo ""
echo "=== Vergleich: NAT GW PIP vs FW PIP ==="
NAT_PIP=$(az network public-ip show -g $RG -n pip-natgw-avd-${PREFIX} --query ipAddress -o tsv)
# vWAN FW hat keine eigene PIP-Ressource in der RG - PIP kommt aus Deployment Output
FW_PIP=$(az deployment group show -g $RG -n main --query properties.outputs.firewallPublicIp.value -o tsv 2>/dev/null || echo "N/A")
echo "NAT Gateway PIP: $NAT_PIP  (WVD/RDP traffic exits here)"
echo "Firewall PIP:    $FW_PIP  (default 0/0 traffic exits here)"
echo ""
echo "Wenn SNATConnectionCount > 0 UND WVD-IPs in User-Routes → RDP geht via NAT GW ✓"

=== WVD Gateway IP-Adressen ===


 / Finished ..
40.64.144.35

=== Effective Route für diese IPs (sollte Source=User, NextHop=Internet sein) ===
[hed ..
  {
    "prefix": "172.183.252.22/32",
    "nextHopType": "Internet",
    "source": "User"
  }
]

=== NAT Gateway SNAT Connections (letzte 30 Min) ===
[
  {
    "time": "2026-06-10T06:11:00Z",
    "connections": 97.0
  },
  {
    "time": "2026-06-10T06:16:00Z",
    "connections": 99.0
  },
  {
    "time": "2026-06-10T06:21:00Z",
    "connections": 95.0
  },
  {
    "time": "2026-06-10T06:26:00Z",
    "connections": 95.0
  },
  {
    "time": "2026-06-10T06:31:00Z",
    "connections": 97.0
  },
  {
    "time": "2026-06-10T06:36:00Z",
    "connections": 102.0
  }
]

=== Vergleich: NAT GW PIP vs FW PIP ===
NAT Gateway PIP: 20.91.239.174  (WVD/RDP traffic exits here)
Firewall PIP:    4.223.154.110  (default 0/0 traffic exits here)

Wenn SNATConnectionCount > 0 UND WVD-IPs in User-Routes → RDP geht via NAT GW ✓


## 7. AVD Required FQDNs testen

Ref: [Required FQDNs for AVD](https://learn.microsoft.com/en-us/azure/virtual-desktop/required-fqdn-endpoint?tabs=azure)

**Erwartung:** Alle Endpoints erreichbar — Firewall Application Rules lassen den Traffic durch.

In [14]:
# HTTPS (443) required endpoints
az vm run-command invoke -g $RG -n $VM --command-id RunPowerShellScript \
  --scripts "@('login.microsoftonline.com','rdweb.wvd.microsoft.com','rdbroker.wvd.microsoft.com','catalogartifact.azureedge.net','wvdportalstorageblob.blob.core.windows.net','aka.ms','graph.microsoft.com') | ForEach-Object { \$r = Test-NetConnection \$_ -Port 443; [PSCustomObject]@{Host=\$_;Connected=\$r.TcpTestSucceeded} } | Format-Table -AutoSize" \
  -o json | tee output/fqdn-https-test.json

{| Finished ..
  "value": [
    {
      "code": "ComponentStatus/StdOut/succeeded",
      "displayStatus": "Provisioning succeeded",
      "level": "Info",
      "message": "Host                                       Connected\n----                                       ---------\nlogin.microsoftonline.com                       True\nrdweb.wvd.microsoft.com                         True\nrdbroker.wvd.microsoft.com                      True\ncatalogartifact.azureedge.net                   True\nwvdportalstorageblob.blob.core.windows.net      True\naka.ms                                          True\ngraph.microsoft.com                             True\n\n"
    },
    {
      "code": "ComponentStatus/StdErr/succeeded",
      "displayStatus": "Provisioning succeeded",
      "level": "Info",
      "message": ""
    }
  ]
}


In [15]:
# HTTP (80) required endpoints (certificates, CRL)
az vm run-command invoke -g $RG -n $VM --command-id RunPowerShellScript \
  --scripts "@('oneocsp.microsoft.com','www.microsoft.com','ctldl.windowsupdate.com','www.msftconnecttest.com') | ForEach-Object { \$r = Test-NetConnection \$_ -Port 80; [PSCustomObject]@{Host=\$_;Connected=\$r.TcpTestSucceeded} } | Format-Table -AutoSize" \
  -o json | tee output/fqdn-http-test.json

{- Finished ..
  "value": [
    {
      "code": "ComponentStatus/StdOut/succeeded",
      "displayStatus": "Provisioning succeeded",
      "level": "Info",
      "message": "Host                    Connected\n----                    ---------\noneocsp.microsoft.com        True\nwww.microsoft.com            True\nctldl.windowsupdate.com      True\nwww.msftconnecttest.com      True\n\n"
    },
    {
      "code": "ComponentStatus/StdErr/succeeded",
      "displayStatus": "Provisioning succeeded",
      "level": "Info",
      "message": ""
    }
  ]
}


## 8. Ergebnis-Auswertung

Prüft die gespeicherten JSON-Dateien und gibt das Testergebnis aus.

In [12]:
echo "=== Deployment Outputs ==="
cat output/deployment-outputs.json | jq '{firewallPublicIp: .firewallPublicIp.value, natGatewayPublicIp: .natGatewayPublicIp.value}'
echo ""
echo "=== User-definierte Routen in Effective Routes ==="
cat output/effective-routes.json | jq '[.value[] | select(.source=="User") | {prefix: .addressPrefix[0], nextHopType, state}]'
echo ""
echo "=== Outbound IP (vom VM) ==="
cat output/outbound-ip.json | jq -r '.value[0].message'

=== Deployment Outputs ===
{
  "firewallPublicIp": "4.223.69.29",
  "natGatewayPublicIp": "20.91.242.226"
}

=== User-definierte Routen in Effective Routes ===
[
  {
    "prefix": "172.183.252.22/32",
    "nextHopType": "Internet",
    "state": "Active"
  },
  {
    "prefix": "10.99.0.0/16",
    "nextHopType": "None",
    "state": "Active"
  }
]

=== Outbound IP (vom VM) ===



---

## AVD Session starten (Browser-Anleitung)

### Voraussetzungen
- User muss der **Application Group** `dag-cptdazavdvwan` zugewiesen sein
- Session Host muss registriert und `Available` sein

### User zur Application Group hinzufügen

In [16]:
# Install extension desktopvirtualization
az extension add --name desktopvirtualization

echo $RG
echo $PREFIX
# UPN anpassen!
USER_UPN="ga1@cptazure.org"
AVD_SCOPE=$(az desktopvirtualization applicationgroup show -g $RG -n dag-${PREFIX} --query id -o tsv)
# User der AVD Desktop Application Group zuweisen
az role assignment create \
  --assignee $USER_UPN \
  --role "Desktop Virtualization User" \
  --scope $AVD_SCOPE

Extension 'desktopvirtualization' 1.0.0 is already installed.
rg-cptdazavdvwan
cptdazavdvwan
{
  "condition": null,
  "conditionVersion": null,
  "createdBy": null,
  "createdOn": "2026-06-10T14:32:53.492387+00:00",
  "delegatedManagedIdentityResourceId": null,
  "description": null,
  "id": "/subscriptions/ff0bb075-6c44-44ee-bb64-d46ce828c62f/resourcegroups/rg-cptdazavdvwan/providers/Microsoft.DesktopVirtualization/applicationgroups/dag-cptdazavdvwan/providers/Microsoft.Authorization/roleAssignments/4107fe5e-2179-40d0-86df-f74d86dc1aab",
  "name": "4107fe5e-2179-40d0-86df-f74d86dc1aab",
  "principalId": "7a4c09e1-dfff-4536-a2c1-f9545e8bdc50",
  "principalType": "User",
  "resourceGroup": "rg-cptdazavdvwan",
  "roleDefinitionId": "/subscriptions/ff0bb075-6c44-44ee-bb64-d46ce828c62f/providers/Microsoft.Authorization/roleDefinitions/1d18fff3-a72a-46b5-b4a9-0b38a3cd7e63",
  "scope": "/subscriptions/ff0bb075-6c44-44ee-bb64-d46ce828c62f/resourcegroups/rg-cptdazavdvwan/providers/Microsoft.De

### Session Host Status prüfen

In [17]:
echo $SUB
echo $RG
echo $PREFIX
az rest --method get \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.DesktopVirtualization/hostPools/hp-${PREFIX}/sessionHosts?api-version=2024-04-03" \
  --query "value[].{name:name,status:properties.status,lastHeartBeat:properties.lastHeartBeat}" -o table

ff0bb075-6c44-44ee-bb64-d46ce828c62f


rg-cptdazavdvwan
cptdazavdvwan
Name                      Status     LastHeartBeat
------------------------  ---------  -----------------------
hp-cptdazavdvwan/vmavd01  Available  2026-06-10T05:53:12.23Z


## Conditional Access Check

Verify which CA policies apply to AVD. Key concerns:
- MFA policies targeting "All Apps" include AVD app IDs
- `signInFrequency: everyTime` forces re-auth on every AVD reconnect
- Managed Identity (RDAgent) is NOT affected by user-scoped CA policies

In [18]:
# List all Conditional Access policies with relevance for AVD
az rest --method get \
  --url "https://graph.microsoft.com/v1.0/identity/conditionalAccess/policies" \
  -o json | jq '[.value[] | {
    displayName,
    state,
    includeApps: .conditions.applications.includeApplications,
    excludeApps: .conditions.applications.excludeApplications,
    includeUsers: .conditions.users.includeUsers,
    grantControls: .grantControls.builtInControls,
    signInFrequency: .sessionControls.signInFrequency
  }]' | tee output/conditional-access.json

[
  {
    "displayName": "Secure password change on high user risk for Microsoft partners and vendors",
    "state": "enabled",
    "includeApps": [
      "All"
    ],
    "excludeApps": [],
    "includeUsers": [
      "All"
    ],
    "grantControls": [
      "mfa",
      "passwordChange"
    ],
    "signInFrequency": null
  },
  {
    "displayName": "Reauthentication on signin risk for Microsoft partners and vendors",
    "state": "enabled",
    "includeApps": [
      "All"
    ],
    "excludeApps": [
      "9cdead84-a844-4324-93f2-b2e6bb768d07"
    ],
    "includeUsers": [
      "All"
    ],
    "grantControls": [
      "mfa"
    ],
    "signInFrequency": {
      "authenticationType": "primaryAndSecondaryAuthentication",
      "frequencyInterval": "everyTime",
      "isEnabled": true,
      "type": null,
      "value": null
    }
  },
  {
    "displayName": "Security info registration for Microsoft partners and vendors",
    "state": "enabled",
    "includeApps": [],
    "excludeApp

In [19]:
# Check if AVD-specific App IDs are explicitly excluded from any CA policy
# AVD App IDs:
#   9cdead84-a844-4324-93f2-b2e6bb768d07 = Windows Virtual Desktop (service)
#   a85cf173-4192-42f8-81fa-777a763e6e2c = Azure Virtual Desktop Client
az rest --method get \
  --url "https://graph.microsoft.com/v1.0/identity/conditionalAccess/policies" \
  -o json | jq '
  .value[] | select(.state == "enabled") |
  {
    displayName,
    avdExcluded: (
      .conditions.applications.excludeApplications |
      any(. == "9cdead84-a844-4324-93f2-b2e6bb768d07" or . == "a85cf173-4192-42f8-81fa-777a763e6e2c")
    ),
    signInFrequency: .sessionControls.signInFrequency.frequencyInterval
  }'

{
  "displayName": "Secure password change on high user risk for Microsoft partners and vendors",
  "avdExcluded": false,
  "signInFrequency": null
}
{
  "displayName": "Reauthentication on signin risk for Microsoft partners and vendors",
  "avdExcluded": true,
  "signInFrequency": "everyTime"
}
{
  "displayName": "Security info registration for Microsoft partners and vendors",
  "avdExcluded": false,
  "signInFrequency": null
}
{
  "displayName": "Multifactor authentication for Microsoft partners and vendors",
  "avdExcluded": false,
  "signInFrequency": null
}


## SSO Configuration

Single sign-on for AVD using Microsoft Entra ID requires 5 steps per
[Microsoft Docs](https://learn.microsoft.com/en-us/azure/virtual-desktop/configure-single-sign-on):

1. **Enable Entra auth for RDP** - Set `isRemoteDesktopProtocolEnabled: true` on Windows Cloud Login SP (`270efc09-cd0d-444b-a71f-39af4910ec45`)
2. **Hide consent prompt** - Create dynamic device group + add as `targetDeviceGroup`
3. **Kerberos Server Object** - Only needed with AD DS (N/A for pure Entra join)
4. **Review Conditional Access** - Exclude AVD App ID from aggressive re-auth policies
5. **Host Pool RDP Property** - Set `enablerdsaadauth:i:1`

### Prerequisites
- Session host must be Entra ID joined (AADLoginForWindows extension with `{"mdmId":""}`)
- FW must allow `enterpriseregistration.windows.net` for device registration
- VM needs SystemAssigned managed identity

In [20]:
# Verify SSO Configuration
echo "=== Step 1: isRemoteDesktopProtocolEnabled ==="
az rest --method get \
  --url "https://graph.microsoft.com/v1.0/servicePrincipals/ecd83fe3-2e79-4cee-9fa9-9b082dcde1bd/remoteDesktopSecurityConfiguration" \
  -o json | jq '{isRemoteDesktopProtocolEnabled, targetDeviceGroups: [.targetDeviceGroups[] | {id, displayName}]}'

echo "=== Step 5: Host Pool RDP Properties ==="
az rest --method get \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.DesktopVirtualization/hostPools/hp-${PREFIX}?api-version=2024-04-03" \
  -o json | jq '.properties.customRdpProperty' | grep -o 'enablerdsaadauth:i:[0-9]'

echo "=== Session Host Status ==="
az rest --method get \
  --url "https://management.azure.com/subscriptions/${SUB}/resourceGroups/${RG}/providers/Microsoft.DesktopVirtualization/hostPools/hp-${PREFIX}/sessionHosts?api-version=2024-04-03" \
  -o json | jq '.value[] | {name, status: .properties.status, lastHeartBeat: .properties.lastHeartBeat}'

=== Step 1: isRemoteDesktopProtocolEnabled ===
{
  "isRemoteDesktopProtocolEnabled": true,
  "targetDeviceGroups": [
    {
      "id": "92322974-f8b4-4762-a7eb-86578b3eb982",
      "displayName": "AVD-SessionHosts-cptdazavdvwan"
    }
  ]
}
=== Step 5: Host Pool RDP Properties ===
enablerdsaadauth:i:1
=== Session Host Status ===
{
  "name": "hp-cptdazavdvwan/vmavd01",
  "status": "Available",
  "lastHeartBeat": "2026-06-10T05:53:12.23Z"
}


### Im Browser verbinden

1. Öffne: **https://client.wvd.microsoft.com/arm/webclient/index.html**
 code --command simpleBrowser.show "https://client.wvd.microsoft.com/arm/webclient/index.html"
2. Anmelden mit deinem Azure AD User (z.B. `chpinoto@...`)
3. Du siehst den Desktop `SessionDesktop` unter dem Workspace `ws-cptdazavdvwan`
4. Klick auf den Desktop → Credentials eingeben → Session startet

**Alternativ** (neuer Client-URL): https://windows.cloud.microsoft

### Troubleshooting

| Problem | Ursache | Lösung |
|---------|---------|--------|
| Kein Desktop sichtbar | User nicht zugewiesen | `az role assignment create` oben ausführen |
| "No session hosts available" | VM nicht registriert | DSC Extension prüfen, Session Host Status |
| Verbindung timeout | FW blockiert RDP/WVD | FW Rules prüfen (Port 443 zu `*.wvd.microsoft.com`) |
| Schwarzer Bildschirm | VM fährt hoch | 1-2 Min warten |